# Spark Optimization 

It includes:

- notes
- performance concepts
- hands-on PySpark examples
- Spark UI debugging steps
- what problem each optimization solves
- experiments to show impact



# Module 1 - Why Spark Optimization Matters

## What is Spark optimization?

Spark optimization means improving the speed and efficiency of a Spark job by reducing:

- unnecessary data movement
- unnecessary recomputation
- poor partition usage
- expensive shuffles
- bad join strategies
- memory pressure

---

## Why do Spark jobs become slow?

Common reasons:

- too many shuffle operations
- bad partition count
- skewed data
- joining huge tables incorrectly
- reading inefficient file formats like CSV/JSON for analytics
- using too many wide transformations
- caching the wrong data
- not filtering early
- collecting too much data to driver

---

## Real-life business impact

If Spark is not optimized:

- jobs take hours instead of minutes
- cloud cost increases
- SLAs fail
- downstream dashboards are delayed
- ML and analytics teams wait for data
- retries and failures increase


## 

Spark is fast when:
- work is parallel
- data movement is low
- partitions are balanced
- joins are smart
- storage format is efficient

Spark becomes slow when:
- one machine gets too much work
- data keeps moving across executors
- shuffle writes and reads become large
- the query plan is inefficient



In [ ]:
# Create a SparkSession if not already available
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("Spark Optimization Teaching Notebook")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
    )

print("Spark session is ready")
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Module 2 - Spark Execution Basics

## Core components

### Driver
The driver is the brain of the application.
It:
- creates SparkSession
- converts your code into a logical plan
- creates DAG
- schedules tasks
- monitors executors

### Executors
Executors are worker processes.
They:
- run tasks
- store cached data
- return results to driver
- handle shuffle read/write

### Cluster Manager
This allocates resources.
Examples:
- YARN
- Kubernetes
- Standalone
- Mesos

---

## Important execution terms

### Job
A job starts when an action is called.

Examples of actions:
- `show()`
- `count()`
- `collect()`
- `write`
- `save`

### Stage
A job is broken into stages.
A stage boundary usually appears when shuffle is required.

### Task
A task is the smallest unit of work.
Each partition generally maps to one task.

### Partition
A partition is a chunk of data processed in parallel.


# Module 3 - Lazy Evaluation

## What is lazy evaluation?

Spark does not execute transformations immediately.

When you write code like:
- filter
- select
- withColumn
- groupBy

Spark only records the plan.

Execution starts only when you call an action.

---

## Why lazy evaluation is useful

Spark can optimize the full pipeline before running it.

That is how Spark:
- removes unnecessary steps
- pushes filters down
- chooses better physical plans
- combines operations when possible





In [ ]:
# Lazy evaluation demo
df = spark.range(1, 1000000)

transformed_df = (
    df.filter("id > 100")
      .withColumnRenamed("id", "num")
      .selectExpr("num", "num * 2 as double_num")
)

print("Nothing is executed yet until an action is called.")
transformed_df.explain(True)

In [ ]:
# This action triggers execution
transformed_df.count()

# Module 4 - DAG and Query Planning

## What is DAG?

DAG stands for Directed Acyclic Graph.

Spark converts your transformations into a DAG.

This helps Spark:
- understand dependencies
- identify parallel work
- split work into stages
- optimize execution

---

## Spark planning layers

### 1. Logical Plan
What you asked Spark to do

### 2. Optimized Logical Plan
Spark rewrites the logical plan using Catalyst Optimizer

### 3. Physical Plan
Spark decides how to actually execute it


Use:
- `df.explain()`
- `df.explain(True)`

This shows:
- parsed plan
- analyzed plan
- optimized logical plan
- physical plan



In [ ]:
# DAG / logical / physical plan demo
sales_df = spark.range(0, 100000).selectExpr(
    "id as order_id",
    "concat('cust_', cast(id % 1000 as string)) as customer_id",
    "cast((id % 20) as int) as category_id",
    "cast((id % 5000) + 100 as double) as amount"
)

optimized_plan_df = (
    sales_df.filter("amount > 1000")
            .groupBy("category_id")
            .sum("amount")
            .orderBy("category_id")
)

optimized_plan_df.explain(True)

# Module 5 - Narrow vs Wide Transformations

## Narrow transformation

A narrow transformation does not require moving data across partitions.

Examples:
- select
- filter
- withColumn
- map
- flatMap on RDD
- union in many cases

### Why narrow transformations are fast
Because each output partition depends on only one input partition.

---

## Wide transformation

A wide transformation requires data reshuffling across partitions.

Examples:
- groupBy
- join
- distinct
- repartition
- orderBy
- reduceByKey at RDD level involves shuffle too

### Why wide transformations are expensive
Because Spark must:
- write shuffle data
- transfer data over network
- read shuffle data again
- synchronize tasks

**Narrow = local work**  
**Wide = data movement**



In [ ]:
from pyspark.sql import functions as F

narrow_df = sales_df.filter("amount > 200").select("order_id", "amount")
wide_df = sales_df.groupBy("category_id").agg(F.sum("amount").alias("total_amount"))

print("Narrow transformation plan:")
narrow_df.explain()

print("\nWide transformation plan:")
wide_df.explain()

# Module 6 - Partitions

## What is a partition?

A partition is a chunk of data.
Spark processes partitions in parallel.

If you have:
- 8 partitions
- 8 available cores

then Spark can process those partitions concurrently.

---

## Why partitioning matters

Too few partitions:
- not enough parallelism
- executors sit idle

Too many partitions:
- too much task overhead
- scheduling overhead increases

Skewed partitions:
- some tasks finish quickly
- one task takes very long
- job waits for slowest task

---

## Important functions

### `repartition(n)`
- increases or redistributes partitions
- causes full shuffle
- useful when balancing data

### `coalesce(n)`
- usually reduces partitions
- avoids full shuffle in many cases
- useful after filtering when you want fewer files

Use `repartition` when you want better balance.  
Use `coalesce` when you want fewer partitions cheaply.



In [ ]:
# Partition inspection
print("Current partitions:", sales_df.rdd.getNumPartitions())

repart_df = sales_df.repartition(10)
coal_df = sales_df.coalesce(2)

print("After repartition:", repart_df.rdd.getNumPartitions())
print("After coalesce:", coal_df.rdd.getNumPartitions())

# Module 7 - Shuffle

## What is shuffle?

Shuffle is the movement of data across executors or partitions so that related records come together.

It happens in operations like:
- groupBy
- join
- orderBy
- distinct
- repartition

---

## Why shuffle is expensive

Shuffle requires:
- writing intermediate data to disk
- sending data over network
- reading data again
- sorting and merging

This makes shuffle one of the biggest performance costs in Spark.

---

## How to reduce shuffle

- filter early
- select only needed columns
- use broadcast joins for small tables
- avoid repeated wide operations
- partition smartly
- reduce unnecessary `distinct`
- use the right shuffle partition count



In [ ]:
# Compare plans with and without extra columns / early filter
wide_input_df = sales_df.withColumn("extra_col", F.expr("amount * 10"))

heavy_df = wide_input_df.groupBy("category_id").agg(F.sum("amount").alias("sum_amount"))
optimized_df = wide_input_df.select("category_id", "amount").filter("amount > 1000") \
                            .groupBy("category_id").agg(F.sum("amount").alias("sum_amount"))

print("Without pruning / early filter:")
heavy_df.explain()

print("\nWith column pruning / early filter:")
optimized_df.explain()

# Module 8 - `spark.sql.shuffle.partitions`

## What is it?

This controls the number of partitions created for shuffle operations in Spark SQL / DataFrame API.

Default is often 200.

---

## Why it matters

If your data is small and shuffle partitions = 200:
- too many tiny tasks
- scheduling overhead increases

If your data is huge and shuffle partitions too low:
- partitions become too large
- memory pressure increases
- long-running tasks appear




In [ ]:
# changing shuffle partition count
print("Before:", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "12")
print("After:", spark.conf.get("spark.sql.shuffle.partitions"))

# Module 9 - Join Optimization

## Why joins are critical

Joins are among the most expensive operations in Spark because they often involve shuffle.

Poor join strategy can:
- increase execution time
- increase memory usage
- cause spills
- create skew problems

---

## Common join strategies

### Broadcast Hash Join
Use when one table is small.
Spark sends the small table to all executors.

Benefits:
- avoids shuffle on large side
- usually very fast

### Sort Merge Join
Common for large-large joins.
Requires shuffle and sorting on both sides.

### Shuffle Hash Join
Can happen for some workloads depending on Spark settings and table sizes.

---

## When to use broadcast join

Broadcast when:
- one table is small enough
- dimension table is small
- lookup or mapping table is small



In [ ]:
# Join optimization
large_orders = spark.range(0, 500000).selectExpr(
    "id as order_id",
    "cast(id % 1000 as int) as customer_key",
    "cast(id % 100 as int) as product_key",
    "cast((id % 5000) + 50 as double) as amount"
)

small_customers = spark.range(0, 1000).selectExpr(
    "id as customer_key",
    "concat('customer_', cast(id as string)) as customer_name",
    "case when id % 2 = 0 then 'active' else 'inactive' end as status"
)

join_normal = large_orders.join(small_customers, "customer_key")
join_broadcast = large_orders.join(F.broadcast(small_customers), "customer_key")

print("Normal join plan:")
join_normal.explain()

print("\nBroadcast join plan:")
join_broadcast.explain()

# Module 10 - Data Skew

## What is data skew?

Data skew happens when some partition gets much more data than others.

Example:
- 90 percent of records have the same key
- one executor gets overloaded
- all others finish quickly
- job waits for one slow task

---

## Symptoms of skew

- one task runs much longer than others
- one partition is much larger
- one executor shows very high input
- stage looks almost complete but one task is still running

---

## Where skew happens

- joins
- groupBy
- aggregation by skewed key
- repartition by skewed column

---

## How to handle skew

1. broadcast small side if possible
2. salting
3. pre-aggregation
4. filtering bad keys if business allows
5. skew hints in Spark 3 for some cases
6. adaptive query execution



In [ ]:
# Create skewed data
skewed_data = [(0, "heavy")] * 50000 + [(i, "light") for i in range(1, 5000)]
skew_df = spark.createDataFrame(skewed_data, ["key", "type"])

print("Skewed data count:", skew_df.count())
skew_df.groupBy("key").count().orderBy(F.desc("count")).show(10, truncate=False)

In [ ]:
# Skewed aggregation plan
skew_agg_df = skew_df.groupBy("key").count()
skew_agg_df.explain()

## Salting concept

Salting means adding artificial randomness to the skewed key so that records get distributed across partitions.

Basic idea:
- Instead of one heavy key
- create heavy_key_0, heavy_key_1, heavy_key_2 ...
- distribute load
- later aggregate again if needed



In [ ]:
# Salting demo
salted_df = skew_df.withColumn("salt", (F.rand() * 10).cast("int")) \
                   .withColumn("salted_key", F.concat_ws("_", F.col("key").cast("string"), F.col("salt").cast("string")))

salted_df.show(5, truncate=False)

# Module 11 - Caching and Persistence

## Why cache?

If the same DataFrame is used multiple times, Spark may recompute it every time unless cached.

Caching stores the result in memory if possible.

---

## When cache helps

Use cache when:
- same DataFrame reused many times
- expensive transformations already done
- iterative algorithms or repeated queries

---

## When cache hurts

Do not cache:
- one-time DataFrames
- very large data that does not fit well
- data that is cheap to recompute

---

## `cache()` vs `persist()`

### `cache()`
Default storage level, generally memory-oriented

### `persist()`
Lets you choose storage level such as:
- MEMORY_ONLY
- MEMORY_AND_DISK
- DISK_ONLY



In [ ]:
from pyspark import StorageLevel

expensive_df = sales_df.filter("amount > 100").withColumn("amount_x2", F.col("amount") * 2)

expensive_df.cache()
print("First action triggers computation and cache population")
expensive_df.count()

print("Second action should reuse cached data")
expensive_df.groupBy("category_id").avg("amount_x2").show()

In [ ]:
# Persistence example
persisted_df = sales_df.filter("amount > 1000").persist(StorageLevel.MEMORY_AND_DISK)
persisted_df.count()
persisted_df.groupBy("category_id").sum("amount").show()
persisted_df.unpersist()

# Module 12 - File Format Optimization

## Why file format matters

Spark analytics is much faster on columnar formats.

---

## CSV / JSON

Problems:
- row-based
- larger storage
- schema inference can be costly
- no efficient column pruning
- poor compression

## Parquet / ORC

Benefits:
- columnar format
- compression
- predicate pushdown
- faster scans
- reads only required columns



In [ ]:
# demo
file_demo_df = sales_df.select("order_id", "category_id", "amount")

base_path = "/mnt/data/spark_opt_demo"  # update for your environment if needed

try:
    file_demo_df.write.mode("overwrite").csv(f"{base_path}/csv_data", header=True)
    file_demo_df.write.mode("overwrite").parquet(f"{base_path}/parquet_data")

    csv_read = spark.read.option("header", True).csv(f"{base_path}/csv_data")
    parquet_read = spark.read.parquet(f"{base_path}/parquet_data")

    print("CSV plan:")
    csv_read.select("amount").explain()

    print("\nParquet plan:")
    parquet_read.select("amount").explain()
except Exception as e:
    print("File demo skipped because the environment path may not exist.")
    print("Error:", e)

# Module 13 - Predicate Pushdown and Column Pruning

## Predicate pushdown

Predicate pushdown means Spark pushes filters to the storage layer so less data is read.

Example:
- filter only amount > 1000
- storage engine skips irrelevant data where possible

---

## Column pruning

Column pruning means Spark reads only the required columns.

Example:
- if you select only `amount`
- Spark does not read all columns from Parquet

---

## Why this improves performance

Less data read means:
- less disk I/O
- less network I/O
- less memory usage
- faster execution



In [ ]:
try:
    parquet_df = spark.read.parquet(f"{base_path}/parquet_data")
    pushdown_df = parquet_df.select("amount").filter("amount > 1000")
    pushdown_df.explain(True)
except Exception as e:
    print("Parquet demo not available in this environment.")
    print("Error:", e)

# Module 14 - Adaptive Query Execution (AQE)

## What is AQE?

Adaptive Query Execution lets Spark optimize the query during runtime based on actual data statistics.

---

## AQE can help with

- coalescing shuffle partitions
- handling skewed joins
- converting sort merge join to broadcast join in some cases
- improving partition sizes dynamically

---

## Why AQE is useful

At planning time Spark estimates.
At runtime Spark knows more.
AQE uses runtime information to improve execution.



In [ ]:
# Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# Module 15 - Small Files Problem

## What is the small files problem?

When Spark writes too many tiny output files:
- metadata overhead increases
- read performance becomes worse
- cloud storage listing becomes slower
- downstream jobs suffer

---

## Why it happens

Usually because:
- too many partitions during write
- high shuffle partition count
- unnecessary repartitioning
- writing partitioned data with very low records per partition

---

## How to reduce it

- use `coalesce()` before write when appropriate
- use better partition strategy
- compact files
- avoid too many output partitions



In [ ]:
# Demonstrate write partition control
small_file_demo = sales_df.filter("amount > 1000").coalesce(2)

try:
    small_file_demo.write.mode("overwrite").parquet(f"{base_path}/compact_output")
    print("Wrote compact output with fewer files")
except Exception as e:
    print("Write skipped in this environment.")
    print("Error:", e)

# Module 16 - Serialization and UDF Considerations

## Why UDFs can hurt performance

Python UDFs often:
- break Catalyst optimization
- increase serialization/deserialization
- move data between JVM and Python
- become slower than built-in functions

---

## Better approach

Prefer:
- built-in Spark SQL functions
- DataFrame expressions
- SQL expressions

Use Python UDF only when necessary.

---

## Teaching line

Built-in functions are usually optimized.
Python UDFs are often a last choice.



In [ ]:
# Built-in function example
builtin_df = sales_df.withColumn("amount_bucket", F.when(F.col("amount") > 2500, "high").otherwise("low"))
builtin_df.show(5)

# Module 17 - Avoiding Common Performance Mistakes

## Mistake 1 - Using `collect()` on huge data
This pulls all data to driver and can crash memory.

## Mistake 2 - Selecting all columns unnecessarily
Always prune columns.

## Mistake 3 - Joining before filtering
Filter early if possible.

## Mistake 4 - Caching everything
Cache only reused expensive data.

## Mistake 5 - Too many `withColumn` chains with heavy logic
Can create complex plans.

## Mistake 6 - Too many `distinct` operations
Distinct is expensive and causes shuffle.

## Mistake 7 - Writing too many small files
This hurts downstream performance.

## Mistake 8 - Ignoring data skew
One slow task can dominate stage time.



# Module 18 - Spark UI Debugging Guide

## Why Spark UI is important

Spark UI helps you answer:
- why is the job slow?
- where is shuffle happening?
- which stage is heavy?
- are tasks balanced?
- are executors spilling memory?
- is there skew?

---

## How to open Spark UI

### Local mode
Usually:
`http://localhost:4040`

### Databricks
Open:
- cluster
- Spark UI
- SQL / jobs / stages tabs

### EMR / YARN
Use the application UI link from the cluster manager

---

## Main tabs in Spark UI

### Jobs tab
Shows:
- all jobs
- duration
- status
- stages per job

Use this to identify which action triggered the job.

### Stages tab
Shows:
- every stage
- number of tasks
- shuffle read/write
- input/output size
- stage duration

Use this to identify expensive stages.

### Tasks view inside a stage
Shows:
- task duration
- executor run time
- input size
- shuffle read/write
- spills
- skew

Use this to identify imbalance.

### Executors tab
Shows:
- executor memory
- disk spills
- task counts
- storage / cached data

Use this to find executor-level bottlenecks.

### SQL tab
Very useful for DataFrame / SQL jobs.
Shows query plan and execution metrics.

---

## What to check first in Spark UI

1. Which job is slow?
2. Which stage inside the job is slow?
3. Is the slow stage a shuffle stage?
4. Are all tasks similar in duration?
5. Is one task much slower than others?
6. Is shuffle read/write high?
7. Are there spills to disk?
8. Are there too many tasks?
9. Are partitions balanced?
10. Is a join causing the issue?



# Module 19 - Spark UI Step-by-Step Debugging Checklist

## Step 1 - Find the slow job
Go to Jobs tab.
Check:
- longest duration
- failed vs successful
- description of action like `count`, `show`, `save`

### Teaching explanation
The slow job tells you where the user-visible problem starts.

---

## Step 2 - Open stage details
Inside the slow job, click the slowest stage.

Check:
- stage duration
- number of tasks
- input size
- shuffle write
- shuffle read

### What it means
Large shuffle often means expensive aggregation or join.

---

## Step 3 - Inspect task distribution
Open the task list for the slow stage.

Look for:
- one task much slower than others
- one task reading much more data
- one task with large shuffle read
- one task spilling to disk

### What it means
This usually indicates skew.

---

## Step 4 - Check executors
Look at Executors tab.

Check:
- task time imbalance
- storage memory used
- disk spills
- failed tasks
- executor lost errors

### What it means
Spills suggest memory pressure.
Repeated failures suggest bad partition size or data issue.

---

## Step 5 - Check SQL tab
If using DataFrames / Spark SQL:
- open SQL tab
- inspect DAG and physical plan
- note sort merge join, broadcast join, exchange nodes

### What to teach
`Exchange` usually indicates shuffle.
Broadcast nodes indicate optimization.

---

## Step 6 - Relate UI back to code
After finding bottleneck:
- too much shuffle -> reduce wide operations / broadcast / repartition smartly
- skew -> salt / AQE / broadcast / pre-aggregate
- many tiny tasks -> tune shuffle partitions
- spills -> increase memory or reduce partition size or optimize joins
- too many files -> coalesce before write



# Module 20 - Live UI Debugging Exercise 1: Aggregation

## Goal
Create a wide aggregation and inspect it in Spark UI.

## What to observe in UI
- one job triggered by action
- stage boundary at shuffle
- shuffle write in map side stage
- shuffle read in reduce side stage
- task count based on partitions



In [ ]:
agg_ui_df = (
    sales_df.repartition(8, "category_id")
            .groupBy("category_id")
            .agg(F.sum("amount").alias("total_amount"),
                 F.avg("amount").alias("avg_amount"),
                 F.count("*").alias("cnt"))
            .orderBy("category_id")
)

agg_ui_df.show()

## UI debugging instructions for this example

1. Run the cell above  
2. Open Spark UI  
3. Go to Jobs tab  
4. Open the latest job  
5. Click the aggregation stage  
6. Inspect:
   - shuffle write
   - shuffle read
   - task duration
7. check:
   - where did the shuffle happen?
   - why is groupBy wide?
   - how many tasks were created?



# Module 21 - Live UI Debugging Exercise 2: Broadcast Join vs Normal Join

## Goal
Compare join strategies using Spark UI and explain plans.

## What to observe
- broadcast join usually avoids large shuffle
- normal large join may show exchange on both sides



In [ ]:
# Normal join action
join_normal.count()

# Broadcast join action
join_broadcast.count()

## UI debugging instructions for join comparison

### For normal join
1. Open the job in Spark UI
2. Inspect stages
3. Check shuffle read/write
4. Go to SQL tab and inspect plan
5. Look for `SortMergeJoin` or `Exchange`

### For broadcast join
1. Open the job
2. Compare number of stages
3. Compare shuffle metrics
4. In SQL tab, look for `BroadcastHashJoin`

### conclusion
Broadcast join reduces data movement when one side is small.



# Module 22 - Live UI Debugging Exercise 3: Skew Detection

## Goal
Create skew and identify it in Spark UI.

## What to observe
- one task much slower
- one task processing more data
- stage almost complete but one task still running



In [ ]:
skew_join_left = skew_df.withColumnRenamed("key", "join_key")
skew_join_right = spark.range(0, 5000).selectExpr("id as join_key", "concat('value_', cast(id as string)) as descr")

skew_join_result = skew_join_left.join(skew_join_right, "join_key", "left")
skew_join_result.count()

## UI debugging instructions for skew

1. Run the skew join cell  
2. Open latest job in Spark UI  
3. Go to stage tasks  
4. Sort tasks by duration  
5. Look for a task that is much slower  
6. Compare input sizes across tasks  

### conclusion
Skew means parallelism looks good on paper, but one partition does most of the work.



# Module 23 - How to Explain Optimization Decisions in Interviews

## Example structure

### 1. State the problem
Our Spark job was slow due to heavy shuffle and skew during join and aggregation.

### 2. Explain what you checked
I checked Spark UI:
- long-running stage
- large shuffle read/write
- skewed tasks
- join type in SQL tab

### 3. Explain root cause
The root cause was:
- a sort merge join on a large dataset
- one key was highly skewed
- shuffle partitions were not tuned

### 4. Explain what you changed
I:
- broadcasted the small dimension table
- filtered early
- reduced unnecessary columns
- enabled AQE
- handled skew with salting
- tuned shuffle partitions

### 5. Explain result
Execution time dropped from X to Y and cloud cost reduced.

---

## Always Rememeber

Optimization is not just changing code.
It is:
- finding bottleneck
- proving root cause
- applying right fix
- validating in Spark UI



# Module 24 - End-to-End Optimization Case Study

## Scenario

An e-commerce pipeline reads raw order data, joins customer lookup, aggregates category sales, and writes output.

### Problems
- source stored in CSV
- join is normal instead of broadcast
- unnecessary columns carried through pipeline
- groupBy creates heavy shuffle
- too many output files

### Optimization approach
1. convert source to Parquet
2. select only required columns
3. filter early
4. broadcast small lookup
5. tune shuffle partitions
6. coalesce before write
7. validate improvements in Spark UI



In [ ]:
# Unoptimized version
unoptimized_pipeline = (
    large_orders.join(small_customers, "customer_key")
                .withColumn("amount_twice", F.col("amount") * 2)
                .groupBy("status", "product_key")
                .agg(F.sum("amount_twice").alias("sales"))
                .orderBy("status", "product_key")
)

# Optimized version
optimized_pipeline_2 = (
    large_orders.select("customer_key", "product_key", "amount")
                .filter("amount > 1000")
                .join(F.broadcast(small_customers.select("customer_key", "status")), "customer_key")
                .groupBy("status", "product_key")
                .agg(F.sum("amount").alias("sales"))
                .coalesce(4)
                .orderBy("status", "product_key")
)

print("Unoptimized plan:")
unoptimized_pipeline.explain()

print("\nOptimized plan:")
optimized_pipeline_2.explain()

In [ ]:
# Trigger both for UI comparison
unoptimized_pipeline.count()
optimized_pipeline_2.count()

## comparison

compare:

### In code
- extra columns removed?
- filter early?
- broadcast added?
- fewer partitions before final output?

### In Spark UI
- fewer shuffle stages?
- lower shuffle read/write?
- faster job duration?
- fewer tasks?
- better task balance?



# Module 25 - Best Practices Summary

## Best practices

1. Filter early  
2. Select only needed columns  
3. Prefer Parquet over CSV for analytics  
4. Use broadcast joins for small tables  
5. Watch shuffle carefully  
6. Tune partition count  
7. Handle skew explicitly  
8. Cache only reused expensive data  
9. Avoid Python UDFs when built-in functions work  
10. Reduce small files problem  
11. Use Spark UI for proof, not guesswork  
12. Enable AQE in modern Spark workloads  

---

## summary

Spark optimization is mainly about:
- minimizing shuffle
- balancing partitions
- choosing efficient join strategy
- reducing data scanned
- avoiding recomputation
- validating performance using Spark UI



# Module 26 - Practice Questions

## Conceptual questions

1. What is the difference between narrow and wide transformations?  
2. Why is shuffle expensive?  
3. What is the difference between repartition and coalesce?  
4. When should you use broadcast join?  
5. What is data skew?  
6. Why can Python UDFs be slower?  
7. What is predicate pushdown?  
8. What is column pruning?  
9. What does AQE do?  
10. Why are small files a problem?

## UI debugging questions

11. In Spark UI, how do you identify skew?  
12. How do you find whether a stage is shuffle-heavy?  
13. What does spill indicate?  
14. Why would one task take much longer than others?  
15. What do you check in SQL tab for join debugging?

## Scenario questions

16. A join is slow and one table is very small. What will you do?  
17. A stage has 1 very slow task and 199 fast tasks. What do you suspect?  
18. A job writes 20,000 tiny Parquet files. How will you reduce this?  
19. Your Spark job reads only one column but still seems slow. What storage checks would you do?  
20. A DataFrame is reused 5 times. What optimization can help?



# Module 27 - Final 

explanation:

Spark optimization is not only about writing smart code.
It is about understanding how Spark executes internally.

When a job is slow, we should not guess.
We should inspect:
- query plan
- partitions
- shuffle
- join type
- skew
- executor behavior
- Spark UI metrics

Then we apply the right solution:
- filter early
- prune columns
- broadcast small tables
- handle skew
- tune partitions
- cache wisely
- write efficient file formats

And finally, we validate the improvement in Spark UI.

That is how optimization is done in real projects.

